# Arrhythmia Detection — Kaggle Pipeline

Runs Models 2 & 3 (1D-CNN, CNN-LSTM) and the GWO/WOA hyperparameter search
for the MIT-BIH arrhythmia classification mini-project. Model 1 (SVM/RF) is
included too so this notebook can run the *entire* project end-to-end if
you want a single artifact, though it also runs fine on a laptop CPU.

**Before running:** open the notebook Settings (right sidebar) and set:
- **Accelerator:** GPU T4 x2 (or P100)
- **Internet:** On

(Requires one-time phone verification on your Kaggle account to unlock
GPU/internet if you haven't done that already.)

Run all cells top to bottom. Total time is roughly 45-75 minutes, dominated
by the GWO/WOA search cell (see that script's own docstring for the
compute-cost breakdown).


## 1. Sanity-check the GPU is visible to TensorFlow

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPUs visible:", tf.config.list_physical_devices("GPU"))


## 2. Install packages Kaggle doesn't preinstall

`tensorflow`, `numpy`, `pandas`, `scikit-learn`, `scipy`, `matplotlib` are already on Kaggle's image. `wfdb`, `PyWavelets`, and `imbalanced-learn` are not.

In [ ]:
!pip install -q wfdb PyWavelets imbalanced-learn


## 3. Write out this project's source files

These cells recreate `src/` under the notebook's writable working directory (`/kaggle/working/`), using the exact files from the project repo (kept in sync — regenerate this notebook from the repo if you change the scripts).

In [ ]:
!mkdir -p src


In [ ]:
%%writefile src/download_data.py
import wfdb 
import os 

os.makedirs('data/raw', exist_ok=True) 
wfdb.dl_database('mitdb', dl_dir='data/raw') 
print('Download complete.')

In [ ]:
%%writefile src/preprocessing.py
"""
preprocessing.py

Shared preprocessing pipeline for the ECG arrhythmia classification project.
Loads MIT-BIH records, filters the signal, segments fixed-length windows around
each annotated beat, maps labels to AAMI EC57 classes, and produces a
patient-independent train/test split.

All three models (SVM/RF baseline, 1D-CNN, CNN-LSTM) should load their data
via build_dataset() (or the saved data/processed/*.npy arrays it produces)
so every model is trained/evaluated on identical data.

Usage:
    python src/preprocessing.py
"""

import os
from pathlib import Path
import numpy as np
import wfdb
from scipy.signal import butter, filtfilt

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(__file__).resolve().parent.parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

FS = 360  # MIT-BIH sampling frequency (Hz)
WINDOW_BEFORE = 90  # samples before R-peak (~0.25s at 360Hz)
WINDOW_AFTER = 110   # samples after R-peak (~0.30s at 360Hz)
# Total window length = WINDOW_BEFORE + WINDOW_AFTER

# Patient-independent split (DS1 / DS2 style split commonly used in MIT-BIH literature)
TRAIN_RECORDS = [
    "101", "106", "108", "109", "112", "114", "115", "116", "118", "119",
    "122", "124", "201", "203", "205", "207", "208", "209", "215", "220",
    "223", "230",
]
TEST_RECORDS = [
    "100", "103", "105", "111", "113", "117", "121", "123", "200", "202",
    "210", "212", "213", "214", "219", "221", "222", "228", "231", "232",
    "233", "234",
]

# AAMI EC57 beat label mapping (MIT-BIH annotation symbol -> AAMI class)
AAMI_MAP = {
    # Normal
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    # Supraventricular ectopic
    "A": "S", "a": "S", "J": "S", "S": "S",
    # Ventricular ectopic
    "V": "V", "E": "V",
    # Fusion
    "F": "F",
    # Unknown / paced
    "/": "Q", "f": "Q", "Q": "Q",
}
CLASS_TO_IDX = {"N": 0, "S": 1, "V": 2, "F": 3, "Q": 4}


# ---------------------------------------------------------------------------
# Filtering
# ---------------------------------------------------------------------------

def bandpass_filter(signal, fs=FS, lowcut=0.5, highcut=40.0, order=3):
    """Apply a Butterworth bandpass filter to remove baseline wander and
    high-frequency noise.

    Args:
        signal: 1D numpy array, raw ECG signal.
        fs: sampling frequency in Hz.
        lowcut, highcut: filter cutoff frequencies in Hz.
        order: filter order.

    Returns:
        1D numpy array, filtered signal (same length as input).
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, signal)


# ---------------------------------------------------------------------------
# Segmentation
# ---------------------------------------------------------------------------

def load_and_segment_record(record_id, data_dir=RAW_DATA_DIR):
    """Load one MIT-BIH record, filter it, and extract fixed-length windows
    around every annotated beat.

    Also returns each kept beat's absolute R-peak sample index (position in
    the original, unfiltered recording). These indices are needed downstream
    to compute RR-interval features correctly -- RR intervals must only be
    computed between consecutive beats *within the same record*, never across
    record boundaries (different records are different half-hour recordings
    from different/unrelated time spans, so a "gap" between the last beat of
    one record and the first beat of the next is meaningless).

    Args:
        record_id: string record name, e.g. '100'.
        data_dir: directory containing the raw .dat/.hea/.atr files.

    Returns:
        windows: numpy array of shape (n_beats, WINDOW_BEFORE + WINDOW_AFTER)
        labels: numpy array of shape (n_beats,) with AAMI class strings
        sample_indices: numpy array of shape (n_beats,) with each beat's
            absolute R-peak sample index within this record (int), in the
            same order as windows/labels.
    """
    record_path = os.path.join(data_dir, record_id)
    record = wfdb.rdrecord(record_path)
    annotation = wfdb.rdann(record_path, "atr")

    # Use channel 0 (MLII lead in most MIT-BIH records)
    signal = record.p_signal[:, 0]
    filtered = bandpass_filter(signal)

    windows = []
    labels = []
    sample_indices = []

    for sample_idx, symbol in zip(annotation.sample, annotation.symbol):
        if symbol not in AAMI_MAP:
            continue  # skip non-beat annotations (rhythm labels, etc.)

        start = sample_idx - WINDOW_BEFORE
        end = sample_idx + WINDOW_AFTER

        # Skip beats too close to the start/end of the recording
        if start < 0 or end > len(filtered):
            continue

        windows.append(filtered[start:end])
        labels.append(AAMI_MAP[symbol])
        sample_indices.append(sample_idx)

    return np.array(windows), np.array(labels), np.array(sample_indices, dtype=np.int64)


def compute_rr_features(sample_indices, fs=FS, local_window=10):
    """Compute RR-interval-derived features for the beats of a single record.

    Must be called separately per record (never on sample indices concatenated
    across multiple records), since RR intervals are only meaningful between
    consecutive beats of the same continuous recording.

    Features per beat (4 columns, all in seconds except the unitless ratio):
        prev_rr:      interval (s) from the previous beat to this beat.
        next_rr:      interval (s) from this beat to the next beat.
        local_avg_rr: mean of the previous `local_window` prev_rr values
                       (early beats use whatever history is available).
        rr_ratio:     prev_rr / next_rr, a standard measure of local rhythm
                       irregularity (close to 1.0 for a regular rhythm).

    Boundary handling: for the first beat in a record there is no previous
    beat, so prev_rr falls back to the first available interval (next_rr);
    symmetrically the last beat's next_rr falls back to the last available
    interval. This avoids NaNs while keeping values within a physiologically
    plausible range, instead of arbitrarily using 0.

    Args:
        sample_indices: 1D numpy array of shape (n_beats,), absolute R-peak
            sample indices for ONE record, in temporal order.
        fs: sampling frequency in Hz.
        local_window: number of preceding beats to average for local_avg_rr.

    Returns:
        numpy array of shape (n_beats, 4): [prev_rr, next_rr, local_avg_rr, rr_ratio]
    """
    sample_indices = np.asarray(sample_indices, dtype=np.float64)
    n = len(sample_indices)

    if n == 0:
        return np.zeros((0, 4))

    if n == 1:
        # Single beat in the record: no neighbouring interval exists at all.
        return np.array([[0.0, 0.0, 0.0, 1.0]])

    rr_intervals = np.diff(sample_indices) / fs  # shape (n-1,)

    prev_rr = np.empty(n)
    next_rr = np.empty(n)
    prev_rr[0] = rr_intervals[0]     # first beat: fall back to first interval
    prev_rr[1:] = rr_intervals
    next_rr[-1] = rr_intervals[-1]   # last beat: fall back to last interval
    next_rr[:-1] = rr_intervals

    local_avg_rr = np.empty(n)
    for i in range(n):
        lo = max(0, i - local_window)
        local_avg_rr[i] = np.mean(prev_rr[lo:i + 1])

    rr_ratio = np.where(next_rr != 0, prev_rr / np.where(next_rr != 0, next_rr, 1.0), 1.0)

    return np.stack([prev_rr, next_rr, local_avg_rr, rr_ratio], axis=1)


def build_dataset(record_ids, data_dir=RAW_DATA_DIR):
    """Build a full dataset (X, y, rr_features) from a list of record IDs.

    RR-interval features are computed per-record (using each record's own
    beat sample indices) before concatenating across records, so values never
    leak across record boundaries.

    Returns:
        X: numpy array of shape (n_total_beats, window_length)
        y: numpy array of shape (n_total_beats,) with integer class indices
        rr_features: numpy array of shape (n_total_beats, 4), see
            compute_rr_features() for column definitions.
    """
    all_windows = []
    all_labels = []
    all_rr_features = []

    for record_id in record_ids:
        try:
            windows, labels, sample_indices = load_and_segment_record(record_id, data_dir)
        except FileNotFoundError:
            print(f"  [skip] record {record_id} not found in {data_dir}")
            continue

        rr_features = compute_rr_features(sample_indices)

        all_windows.append(windows)
        all_labels.append(labels)
        all_rr_features.append(rr_features)
        print(f"  record {record_id}: {len(labels)} beats")

    if not all_windows:
        raise FileNotFoundError(
            f"No records were found in {Path(data_dir).resolve()}. "
            "Check that the MIT-BIH files are downloaded there."
        )

    X = np.concatenate(all_windows, axis=0)
    y_str = np.concatenate(all_labels, axis=0)
    y = np.array([CLASS_TO_IDX[label] for label in y_str])
    rr_features = np.concatenate(all_rr_features, axis=0)

    return X, y, rr_features


# ---------------------------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------------------------

def main():
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

    print("Building training set (patient-independent)...")
    X_train, y_train, rr_train = build_dataset(TRAIN_RECORDS)

    print("\nBuilding test set (patient-independent)...")
    X_test, y_test, rr_test = build_dataset(TEST_RECORDS)

    print(f"\nX_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_test shape:  {X_test.shape},  y_test shape:  {y_test.shape}")

    # Class distribution
    print("\nTrain class distribution:")
    for cls, idx in CLASS_TO_IDX.items():
        count = np.sum(y_train == idx)
        print(f"  {cls}: {count} ({100 * count / len(y_train):.1f}%)")

    print("\nTest class distribution:")
    for cls, idx in CLASS_TO_IDX.items():
        count = np.sum(y_test == idx)
        print(f"  {cls}: {count} ({100 * count / len(y_test):.1f}%)")

    # Save (X/y arrays kept exactly as before for backward compatibility;
    # RR-interval features are saved separately alongside them)
    np.save(os.path.join(PROCESSED_DATA_DIR, "X_train.npy"), X_train)
    np.save(os.path.join(PROCESSED_DATA_DIR, "y_train.npy"), y_train)
    np.save(os.path.join(PROCESSED_DATA_DIR, "X_test.npy"), X_test)
    np.save(os.path.join(PROCESSED_DATA_DIR, "y_test.npy"), y_test)
    np.save(os.path.join(PROCESSED_DATA_DIR, "rr_features_train.npy"), rr_train)
    np.save(os.path.join(PROCESSED_DATA_DIR, "rr_features_test.npy"), rr_test)

    print(f"\nSaved processed arrays (including RR-interval features) to {PROCESSED_DATA_DIR}/")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/svm_baseline.py
"""
svm_baseline.py

Classical ML baseline (SVM / Random Forest) for ECG arrhythmia classification.
Loads the preprocessed beat windows from data/processed/, extracts hand-crafted
features (morphology stats + wavelet coefficients + RR-interval features),
trains SVM and Random Forest classifiers (with light hyperparameter tuning and
SMOTE oversampling of the training set to address severe class imbalance), and
evaluates both on the held-out patient-independent test set.

Usage:
    python src/svm_baseline.py

Requires `imbalanced-learn` (see requirements.txt) for the SMOTE step.
"""

import os
from collections import Counter
import numpy as np
import pywt
from scipy.stats import skew, kurtosis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE

PROCESSED_DATA_DIR = "data/processed"
RESULTS_DIR = "results"
CLASS_NAMES = ["N", "S", "V", "F", "Q"]  # must match CLASS_TO_IDX order in preprocessing.py


# ---------------------------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------------------------

def extract_morphology_features(window):
    """Basic shape statistics of one beat window.

    Args:
        window: 1D numpy array, one beat segment.

    Returns:
        1D numpy array of scalar features.
    """
    return np.array([
        np.max(window),
        np.min(window),
        np.mean(window),
        np.std(window),
        skew(window),
        kurtosis(window),
        np.max(window) - np.min(window),   # peak-to-peak amplitude
        np.sum(np.abs(np.diff(window))),   # rough signal energy/roughness
    ])


def extract_wavelet_features(window, wavelet="db4", level=3):
    """Wavelet decomposition coefficients, summarized to a fixed-length vector.

    Args:
        window: 1D numpy array, one beat segment.
        wavelet: wavelet family to use.
        level: decomposition level.

    Returns:
        1D numpy array of wavelet-derived scalar features (mean + std of each
        coefficient sub-band).
    """
    coeffs = pywt.wavedec(window, wavelet, level=level)
    features = []
    for c in coeffs:
        features.append(np.mean(c))
        features.append(np.std(c))
    return np.array(features)


def extract_features(X, rr_features=None):
    """Build a hand-crafted feature matrix from raw beat windows.

    Combines morphology stats + wavelet coefficients with RR-interval
    features (previous RR, next RR, local average RR, RR ratio) computed
    per-record in preprocessing.py (see `compute_rr_features()` there) and
    saved as rr_features_train.npy / rr_features_test.npy. RR intervals
    capture rhythm-level information (e.g. how early/late a beat arrives
    relative to its neighbours) that pure beat-shape features cannot, and
    are one of the standard feature groups used for AAMI-class ECG beat
    classification.

    Args:
        X: numpy array of shape (n_beats, window_length).
        rr_features: optional numpy array of shape (n_beats, 4) with
            [prev_rr, next_rr, local_avg_rr, rr_ratio] per beat, in the same
            order as X. If None, only morphology + wavelet features are used
            (kept for backward compatibility / quick experiments).

    Returns:
        numpy array of shape (n_beats, n_features).
    """
    feature_rows = []
    for window in X:
        morph = extract_morphology_features(window)
        wave = extract_wavelet_features(window)
        feature_rows.append(np.concatenate([morph, wave]))
    features = np.array(feature_rows)

    if rr_features is not None:
        features = np.concatenate([features, rr_features], axis=1)

    return features


# ---------------------------------------------------------------------------
# Train + evaluate
# ---------------------------------------------------------------------------

def evaluate_model(name, model, X_test, y_test, results_dir=RESULTS_DIR):
    """Run predictions, print + save a classification report and confusion
    matrix to results_dir, matching cnn_model.py / cnn_lstm_model.py's
    evaluate_model() so all models' results land in the same results/
    directory for a direct cross-model comparison.
    """
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(
        y_test, y_pred, target_names=CLASS_NAMES, zero_division=0
    )
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(report)
    print("Confusion matrix:")
    print(cm)

    os.makedirs(results_dir, exist_ok=True)
    safe_name = name.lower().replace(' ', '_').replace('-', '_')
    log_path = os.path.join(results_dir, f"{safe_name}_results.txt")
    with open(log_path, "w") as f:
        f.write(f"{name} results\n")
        f.write(f"Accuracy: {acc:.4f}\n\n")
        f.write(report)
        f.write("\nConfusion matrix (rows=true, cols=pred):\n")
        f.write(str(cm))

    print(f"Saved results to {log_path}")
    return acc


def load_rr_features(split, n_beats):
    """Load RR-interval features for a split ('train'/'test') if present.

    Falls back to None (and prints a warning) if the file doesn't exist yet,
    e.g. because data/processed/ was produced by an older version of
    preprocessing.py -- in that case extract_features() just skips RR
    features so the pipeline still runs end-to-end.
    """
    path = os.path.join(PROCESSED_DATA_DIR, f"rr_features_{split}.npy")
    if not os.path.exists(path):
        print(
            f"  [warn] {path} not found -- re-run src/preprocessing.py to "
            "generate RR-interval features. Continuing without them."
        )
        return None
    rr = np.load(path)
    assert rr.shape[0] == n_beats, (
        f"rr_features_{split}.npy has {rr.shape[0]} rows but X_{split} has {n_beats} beats "
        "-- data/processed/ is likely stale, re-run src/preprocessing.py."
    )
    return rr


def main():
    print("Loading processed data...")
    X_train = np.load(os.path.join(PROCESSED_DATA_DIR, "X_train.npy"))
    y_train = np.load(os.path.join(PROCESSED_DATA_DIR, "y_train.npy"))
    X_test = np.load(os.path.join(PROCESSED_DATA_DIR, "X_test.npy"))
    y_test = np.load(os.path.join(PROCESSED_DATA_DIR, "y_test.npy"))

    rr_train = load_rr_features("train", len(y_train))
    rr_test = load_rr_features("test", len(y_test))

    print(f"Train: {X_train.shape}, Test: {X_test.shape}")

    print("\nExtracting hand-crafted features (morphology + wavelet + RR-interval)...")
    X_train_feat = extract_features(X_train, rr_train)
    X_test_feat = extract_features(X_test, rr_test)
    print(f"Feature matrix shape: {X_train_feat.shape}")

    # Scale features (important for SVM, harmless for RF)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_feat)
    X_test_scaled = scaler.transform(X_test_feat)

    # --- Address severe class imbalance ---
    # class_weight="balanced" alone left S/F/Q with near-zero recall (see
    # results/random_forest_results.txt, results/svm_results.txt from the
    # interim submission): reweighting the loss doesn't give the model more
    # *examples* of the minority classes, which matters a lot for the RBF-SVM
    # decision boundary and for RF splits that rarely see rare classes at all.
    # We apply SMOTE (synthetic minority oversampling) to the TRAINING set
    # only -- never to X_test_scaled/y_test, which must stay a true reflection
    # of the real-world class distribution for evaluation. We still keep
    # class_weight="balanced" on both models as a cheap complementary measure.
    print("\nApplying SMOTE oversampling to the training set only...")
    print("Class counts before SMOTE:", {c: int(np.sum(y_train == i)) for c, i in zip(CLASS_NAMES, range(5))})

    # SMOTE's default sampling_strategy="auto" oversamples every minority
    # class up to the MAJORITY class's count. Here the majority (N) class has
    # tens of thousands of beats, so fully balancing would make the resampled
    # training set roughly 5x the majority count -- impractical to train an
    # RBF-SVM on. Instead we cap how many synthetic samples each minority
    # class is boosted to (still a large increase over the original few
    # hundred/thousand samples for S/F/Q), and leave the majority class alone.
    class_counts = Counter(y_train)
    majority_count = max(class_counts.values())
    target_per_class = min(majority_count, 20000)
    sampling_strategy = {
        cls: max(count, target_per_class)
        for cls, count in class_counts.items()
        if count < majority_count
    }
    smote = SMOTE(random_state=42, k_neighbors=5, sampling_strategy=sampling_strategy)
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
    print("Class counts after SMOTE: ", {c: int(np.sum(y_train_res == i)) for c, i in zip(CLASS_NAMES, range(5))})

    # --- Random Forest (small grid search) ---
    print("\nTuning Random Forest hyperparameters (GridSearchCV)...")
    rf_param_grid = {
        "n_estimators": [200, 400],
        "max_depth": [None, 20],
        "min_samples_leaf": [1, 2],
    }
    # n_jobs=-1 on the *outer* GridSearchCV parallelizes across param/fold
    # combinations; the inner RandomForestClassifier is kept single-threaded
    # (n_jobs=1) to avoid nested parallelism oversubscribing all CPU cores at
    # once, which can badly slow down (or, on some platforms, deadlock) a
    # grid search.
    rf_search = GridSearchCV(
        RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=1),
        rf_param_grid,
        scoring="f1_macro",  # macro-F1 rewards doing well on minority classes, not just accuracy
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
        n_jobs=-1,
    )
    rf_search.fit(X_train_res, y_train_res)
    print(f"Best RF params: {rf_search.best_params_} (cv macro-F1={rf_search.best_score_:.4f})")

    # Refit the final model with the tuned params and full parallelism
    # (no nested GridSearchCV wrapping it anymore, so n_jobs=-1 is safe here).
    rf = RandomForestClassifier(
        class_weight="balanced", random_state=42, n_jobs=-1, **rf_search.best_params_
    )
    rf.fit(X_train_res, y_train_res)
    evaluate_model("Random Forest", rf, X_test_scaled, y_test)

    # --- SVM (small grid search) ---
    # SVC training scales poorly with n_samples, so tune on a stratified
    # subsample of the SMOTE-resampled training set to keep grid search
    # tractable, then fit the chosen params on the full resampled set.
    print("\nTuning SVM hyperparameters (GridSearchCV on a subsample)...")
    rng = np.random.RandomState(42)
    max_tune_samples = 20000
    if len(y_train_res) > max_tune_samples:
        idx = rng.choice(len(y_train_res), size=max_tune_samples, replace=False)
        X_tune, y_tune = X_train_res[idx], y_train_res[idx]
    else:
        X_tune, y_tune = X_train_res, y_train_res

    svm_param_grid = {
        "C": [1, 10],
        "gamma": ["scale", 0.01],
    }
    svm_search = GridSearchCV(
        SVC(kernel="rbf", class_weight="balanced", random_state=42),
        svm_param_grid,
        scoring="f1_macro",
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
        n_jobs=-1,
    )
    svm_search.fit(X_tune, y_tune)
    print(f"Best SVM params: {svm_search.best_params_} (cv macro-F1={svm_search.best_score_:.4f})")

    # RBF-SVM training cost grows super-linearly with n_samples, so even the
    # capped SMOTE output (majority class alone can be tens of thousands of
    # beats) is impractical to fit directly on a laptop. Fit the final SVM on
    # a larger stratified subsample instead of the full resampled set --
    # Random Forest above doesn't need this since tree ensembles scale much
    # better with n_samples.
    max_final_svm_samples = 60000
    if len(y_train_res) > max_final_svm_samples:
        idx = rng.choice(len(y_train_res), size=max_final_svm_samples, replace=False)
        X_svm_fit, y_svm_fit = X_train_res[idx], y_train_res[idx]
    else:
        X_svm_fit, y_svm_fit = X_train_res, y_train_res

    print(f"\nTraining final SVM on {len(y_svm_fit)} (SMOTE-resampled, subsampled) beats...")
    svm = SVC(kernel="rbf", class_weight="balanced", random_state=42, **svm_search.best_params_)
    svm.fit(X_svm_fit, y_svm_fit)
    evaluate_model("SVM", svm, X_test_scaled, y_test)

    print("\nDone. Results saved to results/.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/cnn_model.py
"""
cnn_model.py

Model 2: 1D-CNN trained directly on raw ECG beat windows (no hand-crafted
features), for AAMI 5-class arrhythmia classification (N/S/V/F/Q).

Loads the preprocessed beat windows from data/processed/ (see
src/preprocessing.py for the exact format: X_train/X_test shape
(n_beats, window_length), y as integer class indices 0-4 matching
CLASS_TO_IDX = {"N": 0, "S": 1, "V": 2, "F": 3, "Q": 4}), builds a 1D-CNN
with `build_cnn_model()`, trains it with class weighting to counter the
severe class imbalance (S/F/Q are rare - see results/random_forest_results.txt
and results/svm_results.txt), evaluates on the held-out patient-independent
test set, and saves the trained model + a results log.

--------------------------------------------------------------------------
RUN THIS ON KAGGLE (or any machine with a real TensorFlow+GPU install) -
NOT in this local dev environment.
--------------------------------------------------------------------------
This repo's local environment is Python 3.14 with no stable TensorFlow
wheel available (only a 2.22.0rc0 release candidate) and no GPU, so this
script cannot be executed here. It is written to be uploaded as a Kaggle
notebook/script, where TensorFlow/Keras + GPU come preinstalled.

How to point it at Kaggle's data when you run it there:
    1. Upload data/processed/*.npy (X_train, y_train, X_test, y_test) as a
       Kaggle Dataset, then in the notebook: "Add Data" -> attach it. Kaggle
       mounts it read-only under /kaggle/input/<dataset-name>/.
    2. Either set the environment variable PROCESSED_DATA_DIR before running,
       e.g. (in a notebook cell):
           import os
           os.environ["PROCESSED_DATA_DIR"] = "/kaggle/input/<dataset-name>"
       or just edit the PROCESSED_DATA_DIR constant below directly.
    3. Trained model + logs are written under /kaggle/working/ by default
       when run on Kaggle (MODELS_DIR / LOGS_DIR / RESULTS_DIR also respect
       the PROJECT_ROOT override - see Config section below).

Usage (on Kaggle or a local machine with TensorFlow installed):
    python src/cnn_model.py
"""

import os
import json
from pathlib import Path

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow import keras
from tensorflow.keras import layers

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
# PROJECT_ROOT-relative paths, following the same pattern as preprocessing.py.
# Override via environment variables to point at Kaggle's mounted dataset
# path (e.g. /kaggle/input/<dataset-name>) without touching any logic below.

PROJECT_ROOT = Path(__file__).resolve().parent.parent

PROCESSED_DATA_DIR = Path(os.environ.get(
    "PROCESSED_DATA_DIR", PROJECT_ROOT / "data" / "processed"
))
MODELS_DIR = Path(os.environ.get("MODELS_DIR", PROJECT_ROOT / "models"))
LOGS_DIR = Path(os.environ.get("LOGS_DIR", PROJECT_ROOT / "logs"))
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR", PROJECT_ROOT / "results"))

CLASS_NAMES = ["N", "S", "V", "F", "Q"]  # must match CLASS_TO_IDX order in preprocessing.py
N_CLASSES = len(CLASS_NAMES)

RANDOM_SEED = 42


# ---------------------------------------------------------------------------
# Model definition
# ---------------------------------------------------------------------------

def build_cnn_model(
    input_shape,
    n_classes=N_CLASSES,
    num_filters=(32, 64, 128),
    kernel_size=7,
    pool_size=2,
    dropout_rate=0.3,
    dense_units=64,
    learning_rate=1e-3,
):
    """Build and compile a 1D-CNN for raw ECG beat-window classification.

    This is the shared entry point for both the plain Adam-trained baseline
    run in this file and any later metaheuristic (GWO/WOA) hyperparameter
    search - call it directly with candidate hyperparameter values, it
    returns a freshly built and compiled `keras.Model` each time.

    Architecture: a stack of (Conv1D -> BatchNorm -> ReLU -> MaxPooling1D)
    blocks (one block per entry in `num_filters`), followed by Dropout,
    GlobalAveragePooling1D, a Dense head, and a softmax output layer.

    Args:
        input_shape: tuple, shape of one input sample as (window_length, 1).
            (window_length is WINDOW_BEFORE + WINDOW_AFTER from
            preprocessing.py, e.g. 200; the trailing 1 is the channel dim
            Conv1D expects.)
        n_classes: int, number of output classes (5 for N/S/V/F/Q).
        num_filters: tuple/list of ints, number of filters for each
            successive Conv1D block, e.g. (32, 64, 128) for a 3-block model.
        kernel_size: int, Conv1D kernel size (same for every block).
        pool_size: int, MaxPooling1D pool size (same for every block).
        dropout_rate: float in [0, 1), dropout applied after the conv stack
            and before the dense head.
        dense_units: int, number of units in the dense head layer before
            the final softmax output.
        learning_rate: float, Adam optimizer learning rate.

    Returns:
        A compiled `tensorflow.keras.Model` using
        `sparse_categorical_crossentropy` loss (expects integer-encoded
        labels 0-4 directly, no one-hot conversion needed) and an accuracy
        metric, ready for `model.fit(...)`.
    """
    inputs = keras.Input(shape=input_shape)
    x = inputs

    for filters in num_filters:
        x = layers.Conv1D(filters, kernel_size, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling1D(pool_size=pool_size)(x)

    x = layers.Dropout(dropout_rate)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ---------------------------------------------------------------------------
# Data loading / prep
# ---------------------------------------------------------------------------

def load_data(processed_data_dir=PROCESSED_DATA_DIR):
    """Load the preprocessed train/test arrays produced by preprocessing.py.

    Args:
        processed_data_dir: directory containing X_train.npy, y_train.npy,
            X_test.npy, y_test.npy.

    Returns:
        X_train: (n_train, window_length) float array
        y_train: (n_train,) int array, class indices 0-4
        X_test: (n_test, window_length) float array
        y_test: (n_test,) int array, class indices 0-4
    """
    X_train = np.load(os.path.join(processed_data_dir, "X_train.npy"))
    y_train = np.load(os.path.join(processed_data_dir, "y_train.npy"))
    X_test = np.load(os.path.join(processed_data_dir, "X_test.npy"))
    y_test = np.load(os.path.join(processed_data_dir, "y_test.npy"))
    return X_train, y_train, X_test, y_test


def prepare_inputs(X):
    """Reshape and z-score normalize raw windows for the Conv1D input layer.

    Args:
        X: numpy array of shape (n_beats, window_length).

    Returns:
        numpy array of shape (n_beats, window_length, 1), per-beat
        z-score normalized (each beat window scaled to zero mean / unit
        std) since Conv1D expects a trailing channel dimension.
    """
    X = X.astype("float32")
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, keepdims=True) + 1e-8
    X = (X - mean) / std
    return X[..., np.newaxis]


def compute_class_weights(y_train):
    """Compute balanced class weights from the training label distribution.

    Args:
        y_train: (n_train,) int array of class indices.

    Returns:
        dict mapping class index -> weight, suitable for
        `model.fit(..., class_weight=...)`.
    """
    classes = np.arange(N_CLASSES)
    weights = compute_class_weight(
        class_weight="balanced", classes=classes, y=y_train
    )
    return {int(c): float(w) for c, w in zip(classes, weights)}


# ---------------------------------------------------------------------------
# Train + evaluate
# ---------------------------------------------------------------------------

def evaluate_model(name, model, X_test, y_test, results_dir=RESULTS_DIR):
    """Run predictions, print + save a classification report and confusion
    matrix, mirroring the SVM/RF baseline's evaluate_model() pattern in
    src/svm_baseline.py for a consistent cross-model comparison.
    """
    y_pred_probs = model.predict(X_test)
    y_pred = np.argmax(y_pred_probs, axis=1)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(
        y_test, y_pred, target_names=CLASS_NAMES, zero_division=0
    )
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(report)
    print("Confusion matrix:")
    print(cm)

    os.makedirs(results_dir, exist_ok=True)
    safe_name = name.lower().replace(' ', '_').replace('-', '_')
    log_path = os.path.join(results_dir, f"{safe_name}_results.txt")
    with open(log_path, "w") as f:
        f.write(f"{name} results\n")
        f.write(f"Accuracy: {acc:.4f}\n\n")
        f.write(report)
        f.write("\nConfusion matrix (rows=true, cols=pred):\n")
        f.write(str(cm))

    print(f"Saved results to {log_path}")
    return acc


def train_and_evaluate(
    X_train, y_train, X_test, y_test,
    hyperparams=None,
    epochs=50,
    batch_size=128,
    logs_dir=LOGS_DIR,
    models_dir=MODELS_DIR,
    model_name="cnn_baseline",
):
    """Full train + evaluate pipeline for the 1D-CNN baseline.

    Builds the model via `build_cnn_model()`, trains with class weighting
    and EarlyStopping/ReduceLROnPlateau callbacks, logs per-epoch metrics
    to logs_dir, evaluates on the test set, and saves the trained model.

    Args:
        X_train, y_train, X_test, y_test: preprocessed arrays as returned by
            `load_data()` (raw, not yet reshaped/normalized - this function
            calls `prepare_inputs()` internally).
        hyperparams: dict of keyword args forwarded to `build_cnn_model()`
            (num_filters, kernel_size, dropout_rate, learning_rate, etc.).
            Defaults to `build_cnn_model()`'s own defaults if None - this is
            the "before metaheuristic tuning" baseline configuration.
        epochs: max training epochs (EarlyStopping will typically stop sooner).
        batch_size: training batch size.
        logs_dir: directory for the per-epoch CSV training log.
        models_dir: directory to save the trained model file.
        model_name: base filename (no extension) for the saved model/log.

    Returns:
        (model, history): the trained keras.Model and its History object.
    """
    hyperparams = hyperparams or {}

    X_train_prep = prepare_inputs(X_train)
    X_test_prep = prepare_inputs(X_test)
    input_shape = X_train_prep.shape[1:]  # (window_length, 1)

    print(f"Input shape per sample: {input_shape}")
    model = build_cnn_model(input_shape, n_classes=N_CLASSES, **hyperparams)
    model.summary()

    class_weights = compute_class_weights(y_train)
    print(f"Class weights (balanced): {class_weights}")

    os.makedirs(logs_dir, exist_ok=True)
    os.makedirs(models_dir, exist_ok=True)

    csv_log_path = os.path.join(logs_dir, f"{model_name}_training_log.csv")
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
        ),
        keras.callbacks.CSVLogger(csv_log_path),
    ]

    print(f"\nTraining CNN (per-epoch metrics logged to {csv_log_path})...")
    history = model.fit(
        X_train_prep, y_train,
        validation_data=(X_test_prep, y_test),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=2,
    )

    model_path = os.path.join(models_dir, f"{model_name}.keras")
    model.save(model_path)
    print(f"Saved trained model to {model_path}")

    # Use model_name (not a fixed "CNN") so the baseline run and each
    # metaheuristic-tuned run (see metaheuristic_tuning.py, which calls this
    # with model_name="gwo_tuned_cnn" / "woa_tuned_cnn") write to distinct
    # results files instead of all overwriting results/cnn_results.txt.
    evaluate_model(model_name, model, X_test_prep, y_test)

    return model, history


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    keras.utils.set_random_seed(RANDOM_SEED)

    print(f"Loading processed data from {PROCESSED_DATA_DIR} ...")
    X_train, y_train, X_test, y_test = load_data()
    print(f"Train: {X_train.shape}, Test: {X_test.shape}")

    # Baseline hyperparameters (before metaheuristic/GWO-WOA tuning). A
    # teammate's optimizer script should call build_cnn_model() directly
    # with its own candidate hyperparameter dicts instead of editing this.
    baseline_hyperparams = dict(
        num_filters=(32, 64, 128),
        kernel_size=7,
        pool_size=2,
        dropout_rate=0.3,
        dense_units=64,
        learning_rate=1e-3,
    )

    train_and_evaluate(
        X_train, y_train, X_test, y_test,
        hyperparams=baseline_hyperparams,
        epochs=50,
        batch_size=128,
        model_name="cnn_baseline",
    )

    print("\nDone. Preliminary CNN baseline results saved to results/ and logs/.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/cnn_lstm_model.py
"""
cnn_lstm_model.py

Model 3: CNN-LSTM hybrid for ECG arrhythmia classification.

Unlike the plain 1D-CNN (Model 2), which classifies each beat window in
isolation, this model captures BEAT-TO-BEAT TEMPORAL dependencies: consecutive
beats from the same patient record are grouped into short sequences, a
Conv1D+pooling block extracts per-beat morphology features (applied to every
beat in the sequence via TimeDistributed), and an LSTM then models how those
per-beat feature vectors evolve across the sequence. A dense head classifies
the beat at the END of each sequence (the "current" beat, given its recent
history) into one of the 5 AAMI classes.

Why classify the last beat rather than the whole sequence at once:
preprocessing.py's labels are per-beat, and the classical approach in
arrhythmia literature (RR-interval-aware classifiers) is to predict the
current beat conditioned on a short window of preceding beats. Predicting
every beat in the sequence in parallel would work too, but would require a
many-to-many output head and repeats every beat's label ~seq_len times in the
loss, which is more complex for a mini-project with modest benefit. Using the
last beat also mirrors how this model would be used online: at each new beat,
look back at the last few beats and classify the newest one.

Sequence construction (see build_beat_sequences() / build_sequence_dataset()):
preprocessing.build_dataset() concatenates beats from all records into one
flat (n_beats, window_length) array, which loses each record's beat order and
would let a sliding window span two different patients. This file re-derives
per-record beat order by calling preprocessing.load_and_segment_record() once
per record (same function the shared pipeline already uses), and slides a
window of `seq_len` consecutive beats (default 5) over each record
independently, stepping by 1 beat, using the LAST beat's label as the
sequence's target. Sequences are only ever built from beats within a single
record, so no sequence mixes two patients.

Resulting shapes:
    X_seq: (n_sequences, seq_len, window_length)   float32
    y_seq: (n_sequences,)                          int labels 0-4 (AAMI order N,S,V,F,Q)

Architecture (build_cnn_lstm_model):
    Input (seq_len, window_length, 1)
      -> TimeDistributed(Conv1D -> BatchNorm -> ReLU -> MaxPool1D)  [x n_conv_blocks]
      -> TimeDistributed(GlobalAveragePooling1D)     # per-beat feature vector
      -> LSTM(lstm_units)                             # models across the beat sequence
      -> Dropout
      -> Dense(dense_units, relu)
      -> Dense(n_classes, softmax)                    # classifies the LAST beat

IMPORTANT — WHERE TO RUN THIS:
This script requires `tensorflow.keras` with a working TensorFlow install and
is intended to run on Kaggle (or any machine with GPU + a stable TensorFlow),
NOT in this project's local dev environment. The local machine only has
Python 3.14, for which no stable TensorFlow release exists yet (only a
2.22.0rc0 release candidate), and there is no local GPU. Do not try to `pip
install tensorflow` and run this locally — write/edit the code here, then
upload/run it as a Kaggle notebook or script.

Adapting the data path for Kaggle:
- Locally, data is read from PROJECT_ROOT/data/raw (raw MIT-BIH .dat/.hea/.atr
  files) per preprocessing.py's layout.
- On Kaggle, add the MIT-BIH dataset via "Add Input" in the notebook's data
  panel; it will be mounted read-only under /kaggle/input/<dataset-slug>/...
- Point this script at the right location by setting the RAW_DATA_DIR
  environment variable before running, e.g. in a Kaggle notebook cell:
      import os
      os.environ["RAW_DATA_DIR"] = "/kaggle/input/mitbih-arrhythmia/raw"
  or just edit the RAW_DATA_DIR constant below directly. Trained models and
  logs are written under MODELS_DIR / LOGS_DIR, which default to
  /kaggle/working/{models,logs} style relative paths when run from a Kaggle
  working directory, or PROJECT_ROOT-relative paths locally -- override with
  the MODELS_DIR / LOGS_DIR env vars if needed.

Usage (on Kaggle / a GPU+TensorFlow machine):
    python src/cnn_lstm_model.py
"""

import os
from pathlib import Path

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# preprocessing.py holds the shared record lists, AAMI label mapping, and the
# per-record loading function we reuse to rebuild per-record beat order.
from preprocessing import (
    TRAIN_RECORDS,
    TEST_RECORDS,
    CLASS_TO_IDX,
    load_and_segment_record,
)

# ---------------------------------------------------------------------------
# Config -- override any of these via environment variables so the same
# script works unchanged locally (paths relative to the project root) and on
# Kaggle (paths under /kaggle/input or /kaggle/working).
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(__file__).resolve().parent.parent

# Raw MIT-BIH files. On Kaggle, set RAW_DATA_DIR to wherever the dataset was
# added under /kaggle/input/, e.g. "/kaggle/input/mitbih-arrhythmia/raw".
RAW_DATA_DIR = Path(os.environ.get("RAW_DATA_DIR", PROJECT_ROOT / "data" / "raw"))

MODELS_DIR = Path(os.environ.get("MODELS_DIR", PROJECT_ROOT / "models"))
LOGS_DIR = Path(os.environ.get("LOGS_DIR", PROJECT_ROOT / "logs"))
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR", PROJECT_ROOT / "results"))

CLASS_NAMES = ["N", "S", "V", "F", "Q"]  # must match CLASS_TO_IDX order in preprocessing.py

# Beats per sequence and the sliding-window step (in beats) used to build
# sequences. seq_len=5 with step=1 means every beat (once enough history has
# accumulated) becomes the target of one sequence, looking back 4 beats.
DEFAULT_SEQ_LEN = 5
DEFAULT_STEP = 1


# ---------------------------------------------------------------------------
# Sequence construction (respects per-record boundaries)
# ---------------------------------------------------------------------------

def build_beat_sequences(windows, labels, seq_len=DEFAULT_SEQ_LEN, step=DEFAULT_STEP):
    """Slide a window of `seq_len` consecutive beats over ONE record's beats.

    Args:
        windows: numpy array of shape (n_beats, window_length), beats from a
            single record, in the original chronological order they were
            annotated in.
        labels: numpy array of shape (n_beats,), integer class indices
            aligned with `windows`.
        seq_len: number of consecutive beats per sequence.
        step: how many beats to advance the window start by each step.

    Returns:
        X_seq: numpy array of shape (n_sequences, seq_len, window_length)
        y_seq: numpy array of shape (n_sequences,) -- label of the LAST beat
            in each sequence (the beat being classified).
    """
    n_beats = len(windows)
    X_seq, y_seq = [], []
    for start in range(0, n_beats - seq_len + 1, step):
        end = start + seq_len
        X_seq.append(windows[start:end])
        y_seq.append(labels[end - 1])  # label of the last (most recent) beat
    if not X_seq:
        window_length = windows.shape[1] if windows.ndim == 2 else 0
        return (
            np.empty((0, seq_len, window_length), dtype=np.float32),
            np.empty((0,), dtype=np.int64),
        )
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.int64)


def build_sequence_dataset(record_ids, data_dir=RAW_DATA_DIR, seq_len=DEFAULT_SEQ_LEN,
                            step=DEFAULT_STEP):
    """Build a full beat-sequence dataset (X, y) from a list of record IDs.

    Mirrors preprocessing.build_dataset(), but instead of concatenating all
    beats into one flat array (which would lose each record's beat order and
    let a sequence span two different patients), this loads each record
    separately, builds sliding-window beat sequences WITHIN that record only,
    and then concatenates the resulting sequences across records.

    Args:
        record_ids: list of MIT-BIH record ID strings (e.g. TRAIN_RECORDS).
        data_dir: directory containing the raw .dat/.hea/.atr files.
        seq_len: beats per sequence (see build_beat_sequences).
        step: sliding-window step in beats (see build_beat_sequences).

    Returns:
        X: numpy array of shape (n_sequences, seq_len, window_length)
        y: numpy array of shape (n_sequences,) with integer class indices (0-4)
    """
    all_X, all_y = [], []

    for record_id in record_ids:
        try:
            windows, label_strs, _sample_indices = load_and_segment_record(record_id, data_dir)
        except FileNotFoundError:
            print(f"  [skip] record {record_id} not found in {data_dir}")
            continue

        if len(label_strs) < seq_len:
            print(f"  [skip] record {record_id}: only {len(label_strs)} beats (< seq_len={seq_len})")
            continue

        label_idx = np.array([CLASS_TO_IDX[lbl] for lbl in label_strs])
        X_seq, y_seq = build_beat_sequences(windows, label_idx, seq_len=seq_len, step=step)

        all_X.append(X_seq)
        all_y.append(y_seq)
        print(f"  record {record_id}: {len(label_strs)} beats -> {len(y_seq)} sequences")

    if not all_X:
        raise FileNotFoundError(
            f"No records were found in {Path(data_dir).resolve()}. "
            "Check that the MIT-BIH files are downloaded there, or set the "
            "RAW_DATA_DIR environment variable (see this file's module docstring)."
        )

    X = np.concatenate(all_X, axis=0)
    y = np.concatenate(all_y, axis=0)
    return X, y


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------

def build_cnn_lstm_model(
    input_shape,
    n_classes=5,
    conv_filters=(32, 64),
    conv_kernel_size=5,
    pool_size=2,
    lstm_units=64,
    dense_units=64,
    dropout_rate=0.3,
    learning_rate=1e-3,
):
    """Build the CNN-LSTM hybrid model.

    Args:
        input_shape: tuple (seq_len, window_length, n_channels), the shape of
            one input sequence (n_channels is 1 for the raw single-lead ECG
            windows produced by build_sequence_dataset -- add a trailing
            channel axis before feeding data in, see main()).
        n_classes: number of output classes (5 for N/S/V/F/Q).
        conv_filters: tuple of filter counts, one Conv1D+pool block per entry.
        conv_kernel_size: kernel size for every Conv1D block.
        pool_size: pool size for every MaxPool1D block.
        lstm_units: number of units in the LSTM layer that runs across beats.
        dense_units: number of units in the dense layer before the output.
        dropout_rate: dropout applied after the LSTM and after the dense layer.
        learning_rate: Adam learning rate.

    Returns:
        A compiled tensorflow.keras.Model. Input shape (batch, seq_len,
        window_length, n_channels); output shape (batch, n_classes) softmax
        probabilities for the LAST beat in each input sequence.
    """
    from tensorflow.keras import layers, models, optimizers

    inputs = layers.Input(shape=input_shape, name="beat_sequence")
    x = inputs

    # Per-beat convolutional feature extraction, applied identically to every
    # beat in the sequence (shared weights across the time axis).
    for n_filters in conv_filters:
        x = layers.TimeDistributed(
            layers.Conv1D(n_filters, conv_kernel_size, padding="same")
        )(x)
        x = layers.TimeDistributed(layers.BatchNormalization())(x)
        x = layers.TimeDistributed(layers.Activation("relu"))(x)
        x = layers.TimeDistributed(layers.MaxPooling1D(pool_size=pool_size))(x)

    # Collapse each beat's conv feature map to one feature vector per beat.
    x = layers.TimeDistributed(layers.GlobalAveragePooling1D())(x)

    # Model temporal dependencies ACROSS the beat sequence.
    x = layers.LSTM(lstm_units)(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(n_classes, activation="softmax", name="beat_class")(x)

    model = models.Model(inputs, outputs, name="cnn_lstm_hybrid")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ---------------------------------------------------------------------------
# Train + evaluate
# ---------------------------------------------------------------------------

def evaluate_model(name, model, X_test, y_test, results_dir=RESULTS_DIR):
    """Run predictions, print + save a classification report and confusion
    matrix to results_dir (RESULTS_DIR), matching cnn_model.py's
    evaluate_model() so all models' final results land in the same results/
    directory for a direct cross-model comparison. Per-epoch training
    metrics (loss/accuracy per epoch) are logged separately to LOGS_DIR via
    CSVLogger in main() -- this function only writes the final test-set
    evaluation.
    """
    y_prob = model.predict(X_test)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(
        y_test, y_pred, target_names=CLASS_NAMES, zero_division=0
    )
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(report)
    print("Confusion matrix:")
    print(cm)

    os.makedirs(results_dir, exist_ok=True)
    safe_name = name.lower().replace(' ', '_').replace('-', '_')
    log_path = os.path.join(results_dir, f"{safe_name}_results.txt")
    with open(log_path, "w") as f:
        f.write(f"{name} results\n")
        f.write(f"Accuracy: {acc:.4f}\n\n")
        f.write(report)
        f.write("\nConfusion matrix (rows=true, cols=pred):\n")
        f.write(str(cm))

    print(f"Saved results to {log_path}")
    return acc


def main():
    # Imported here (not at module level) so this file can still be read /
    # imported for its data-prep helpers (build_sequence_dataset etc.) in an
    # environment without TensorFlow installed -- only main() actually needs
    # tensorflow.keras.
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, CSVLogger

    seq_len = int(os.environ.get("SEQ_LEN", DEFAULT_SEQ_LEN))
    step = int(os.environ.get("SEQ_STEP", DEFAULT_STEP))

    print(f"Building CNN-LSTM sequence dataset (seq_len={seq_len}, step={step})...")
    print("Training set (patient-independent):")
    X_train, y_train = build_sequence_dataset(TRAIN_RECORDS, RAW_DATA_DIR, seq_len, step)
    print("\nTest set (patient-independent):")
    X_test, y_test = build_sequence_dataset(TEST_RECORDS, RAW_DATA_DIR, seq_len, step)

    print(f"\nX_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_test shape:  {X_test.shape},  y_test shape:  {y_test.shape}")

    # Add a trailing channel axis: Conv1D expects (steps, channels) per beat,
    # so each sequence element goes from (window_length,) to (window_length, 1).
    X_train = X_train[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    # Class distribution (S/F/Q are rare -- see results/random_forest_results.txt
    # and results/svm_results.txt).
    print("\nTrain class distribution:")
    for cls, idx in CLASS_TO_IDX.items():
        count = np.sum(y_train == idx)
        print(f"  {cls}: {count} ({100 * count / len(y_train):.1f}%)")

    # Handle class imbalance the same way as the baseline models: balanced
    # class weights passed into model.fit() rather than resampling the data.
    class_weight_arr = compute_class_weight(
        class_weight="balanced", classes=np.arange(len(CLASS_TO_IDX)), y=y_train
    )
    class_weight = {i: w for i, w in enumerate(class_weight_arr)}
    print(f"\nClass weights: {class_weight}")

    input_shape = X_train.shape[1:]  # (seq_len, window_length, 1)
    model = build_cnn_lstm_model(input_shape, n_classes=len(CLASS_TO_IDX))
    model.summary()

    os.makedirs(LOGS_DIR, exist_ok=True)
    os.makedirs(MODELS_DIR, exist_ok=True)

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
        # Per-epoch metrics logged to logs/, per the project convention of
        # logging metrics per epoch for the interim/final report.
        CSVLogger(os.path.join(LOGS_DIR, "cnn_lstm_training_log.csv")),
    ]

    print("\nTraining CNN-LSTM hybrid...")
    model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=int(os.environ.get("EPOCHS", 50)),
        batch_size=int(os.environ.get("BATCH_SIZE", 64)),
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=2,
    )

    evaluate_model("CNN-LSTM", model, X_test, y_test)

    model_path = os.path.join(MODELS_DIR, "cnn_lstm_model.keras")
    model.save(model_path)
    print(f"\nSaved trained model to {model_path}")

    print("\nDone. CNN-LSTM hybrid results saved to results/.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/metaheuristic_tuning.py
"""
metaheuristic_tuning.py

Side experiment (see .github/copilot-instructions.md): does metaheuristic
optimization -- Grey Wolf Optimizer (GWO, Mirjalili et al. 2014) and Whale
Optimization Algorithm (WOA, Mirjalili & Lewis 2016) -- find better 1D-CNN
hyperparameters than the standard Adam-trained baseline in src/cnn_model.py?

This script does NOT redefine the CNN. It imports and calls the exact
functions already written in src/cnn_model.py:
    build_cnn_model(input_shape, n_classes=5, num_filters=(32,64,128),
                     kernel_size=7, pool_size=2, dropout_rate=0.3,
                     dense_units=64, learning_rate=1e-3)
    load_data(), prepare_inputs(X), compute_class_weights(y_train),
    evaluate_model(name, model, X_test, y_test),
    train_and_evaluate(X_train, y_train, X_test, y_test, hyperparams=None, ...)
Every candidate solution produced by GWO/WOA is turned into a hyperparams
dict and fed straight into build_cnn_model()/train_and_evaluate() -- nothing
here reimplements the model architecture.

--------------------------------------------------------------------------
RUN THIS ON KAGGLE (or any machine with a real TensorFlow+GPU install) -
NOT in this local dev environment.
--------------------------------------------------------------------------
Same constraint as cnn_model.py: this repo's local environment is Python
3.14 with no working stable TensorFlow wheel and no GPU. This script is
written to be uploaded to Kaggle alongside cnn_model.py (same directory,
same data-loading conventions) and run there.

Expected compute cost (read this before hitting Run on Kaggle):
    Each fitness evaluation trains one full CNN for a few epochs. With the
    settings chosen below:
        POP_SIZE = 6 candidates, MAX_ITER = 6 iterations per optimizer,
        FITNESS_EPOCHS = 8 epochs per candidate (with early stopping, so
        often fewer in practice).
    Total fitness-evaluation trainings per optimizer = POP_SIZE * MAX_ITER
        = 6 * 6 = 36 CNN trainings (up to 8 epochs each).
    Running BOTH GWO and WOA = 72 short CNN trainings.
    Plus 2 final "retrain the best config on the full training set" runs
    (FINAL_EPOCHS = 50 each, same as the baseline in cnn_model.py).
    On a Kaggle T4/P100 GPU, each short fitness-evaluation training of this
    small 1D-CNN on MIT-BIH beat windows typically takes well under a
    minute; the two final full retrains are comparable in cost to running
    cnn_model.py's own baseline twice. Budget roughly 30-60 minutes total
    for the whole script, but this scales directly with POP_SIZE, MAX_ITER,
    FITNESS_EPOCHS and dataset size -- reduce those first if you are close
    to Kaggle's session time limit.

How to point it at Kaggle's data: identical to cnn_model.py -- set
PROCESSED_DATA_DIR (and optionally MODELS_DIR/LOGS_DIR/RESULTS_DIR) env
vars before running, or edit the constants directly.

Usage (on Kaggle or a local machine with TensorFlow installed):
    python src/metaheuristic_tuning.py
"""

import os
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from tensorflow import keras

# cnn_model.py lives next to this file in src/ -- import its exact public
# interface directly rather than re-implementing anything here. When this
# script is run as `python src/metaheuristic_tuning.py` (or uploaded next
# to cnn_model.py on Kaggle), the script's own directory is already on
# sys.path, but we add it defensively in case this is imported differently.
sys.path.insert(0, str(Path(__file__).resolve().parent))

from cnn_model import (  # noqa: E402
    PROCESSED_DATA_DIR,
    MODELS_DIR,
    LOGS_DIR,
    RESULTS_DIR,
    CLASS_NAMES,
    N_CLASSES,
    RANDOM_SEED,
    build_cnn_model,
    load_data,
    prepare_inputs,
    compute_class_weights,
    evaluate_model,
    train_and_evaluate,
)

# ---------------------------------------------------------------------------
# Config: metaheuristic search settings
# ---------------------------------------------------------------------------
# Deliberately small -- see the module docstring above for the compute-cost
# accounting. This is a mini-project demonstration of GWO/WOA, not a
# research-grade hyperparameter search.
POP_SIZE = 6          # wolves / whales
MAX_ITER = 6          # optimizer iterations
FITNESS_EPOCHS = 8    # epochs per candidate during search (early stopping applies)
FITNESS_PATIENCE = 3  # EarlyStopping patience during the (short) fitness runs
FINAL_EPOCHS = 50     # epochs for the final retrain-on-full-data run (matches baseline)
BATCH_SIZE = 128
VAL_FRACTION = 0.15   # carved out of the TRAINING set only, never the test set

RANDOM_STATE = RANDOM_SEED

# ---------------------------------------------------------------------------
# Hyperparameter search space
# ---------------------------------------------------------------------------
# Each optimizer works with a real-valued position vector of length 5. The
# dimensions and their (lower, upper) bounds are:
#
#   idx  name               bounds           meaning / decoding
#   ---  -----------------  ---------------  --------------------------------
#   0    log10_lr           (-4.0, -2.0)     learning_rate = 10**log10_lr
#                                             -> continuous, searched on a
#                                             log scale, i.e. lr in [1e-4, 1e-2]
#   1    dropout_rate       (0.1, 0.5)       used directly (continuous)
#   2    num_filters_base   (16, 128)        rounded to nearest int; the 3
#                                             conv blocks use
#                                             (base, base*2, base*4) filters,
#                                             mirroring cnn_model.py's default
#                                             (32, 64, 128) shape
#   3    dense_units        (32, 256)        rounded to nearest int
#   4    kernel_size        (3, 11)          rounded to nearest int, then
#                                             nudged to the nearest odd
#                                             number (odd kernels keep
#                                             "same" padding symmetric) and
#                                             re-clipped to [3, 11]
#
# GWO/WOA both operate on these real-valued vectors and clip back into
# BOUNDS after every position update; decode_position() below is the single
# place that maps a raw vector to the keyword args build_cnn_model() expects.

BOUNDS = np.array([
    (-4.0, -2.0),   # log10 learning rate
    (0.1, 0.5),     # dropout_rate
    (16.0, 128.0),  # num_filters_base
    (32.0, 256.0),  # dense_units
    (3.0, 11.0),    # kernel_size
])
DIM = BOUNDS.shape[0]
LOWER = BOUNDS[:, 0]
UPPER = BOUNDS[:, 1]


def decode_position(pos):
    """Map a raw continuous position vector to a build_cnn_model() kwargs dict.

    Args:
        pos: array-like of length DIM (see BOUNDS table above for the
            per-dimension meaning and bounds). Not assumed to already be
            clipped to bounds -- this function clips defensively.

    Returns:
        dict with keys learning_rate, dropout_rate, num_filters, dense_units,
        kernel_size, ready to pass as build_cnn_model(**hyperparams) /
        train_and_evaluate(..., hyperparams=hyperparams).
    """
    pos = np.clip(np.asarray(pos, dtype=float), LOWER, UPPER)

    learning_rate = float(10.0 ** pos[0])
    dropout_rate = float(pos[1])

    filters_base = int(round(pos[2]))
    dense_units = int(round(pos[3]))

    kernel_size = int(round(pos[4]))
    if kernel_size % 2 == 0:
        kernel_size += 1  # keep kernels odd for symmetric "same" padding
    kernel_size = int(np.clip(kernel_size, BOUNDS[4, 0], BOUNDS[4, 1]))
    if kernel_size % 2 == 0:  # clipping could re-introduce an even number at the edge
        kernel_size -= 1

    num_filters = (filters_base, filters_base * 2, filters_base * 4)

    return dict(
        learning_rate=learning_rate,
        dropout_rate=dropout_rate,
        num_filters=num_filters,
        dense_units=dense_units,
        kernel_size=kernel_size,
    )


def random_population(pop_size, rng):
    """Uniformly sample `pop_size` positions within BOUNDS.

    Returns an (pop_size, DIM) array.
    """
    return rng.uniform(LOWER, UPPER, size=(pop_size, DIM))


# ---------------------------------------------------------------------------
# Fitness function
# ---------------------------------------------------------------------------

def make_fitness_fn(X_tr, y_tr, X_val, y_val, input_shape):
    """Build a fitness(position) -> macro-F1 closure over a fixed train/val split.

    The validation split is carved out of the TRAINING data only (see
    main()); the real held-out test set from cnn_model.load_data() is never
    touched during the search, so hyperparameters are not tuned against it.

    Macro-F1 (rather than accuracy) is used as the fitness because the AAMI
    classes are heavily imbalanced (S/F/Q are rare -- see
    results/svm_results.txt / results/random_forest_results.txt), so a
    trivial "always predict N" model would score high accuracy but ~0
    macro-F1. Maximizing macro-F1 rewards candidates that actually learn the
    rare classes, which is the point of tuning in the first place.
    """
    class_weights = compute_class_weights(y_tr)

    def fitness(position):
        hyperparams = decode_position(position)
        model = build_cnn_model(input_shape, n_classes=N_CLASSES, **hyperparams)

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=FITNESS_PATIENCE,
                restore_best_weights=True,
            ),
        ]
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=FITNESS_EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=class_weights,
            callbacks=callbacks,
            verbose=0,
        )

        y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
        macro_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)

        # Free GPU/CPU memory between candidates -- important since we build
        # a fresh model per fitness evaluation, many times per optimizer run.
        keras.backend.clear_session()

        return float(macro_f1), hyperparams

    return fitness


# ---------------------------------------------------------------------------
# Grey Wolf Optimizer (GWO) -- Mirjalili, Mirjalili & Lewis, 2014
# ---------------------------------------------------------------------------

def grey_wolf_optimizer(fitness_fn, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=RANDOM_STATE):
    """Minimal Grey Wolf Optimizer over a continuous, bounded search space.

    Standard GWO: a pack of `pop_size` wolves searches for prey (the
    optimum). The three best wolves each iteration are labelled alpha, beta,
    delta; every wolf (including alpha/beta/delta themselves) then moves
    towards the average of the positions those three "leaders" imply,
    with the exploration/exploitation balance controlled by the coefficient
    `a`, which decays linearly from 2 to 0 over the iterations.

    Args:
        fitness_fn: callable position -> (score, hyperparams_dict), where
            higher score is better (we MAXIMIZE macro-F1 here, so GWO's
            usual "minimize" convention is flipped by tracking the largest
            score as alpha/beta/delta instead of the smallest).
        pop_size: number of wolves.
        max_iter: number of iterations.
        seed: RNG seed for reproducibility.

    Returns:
        (best_position, best_score, best_hyperparams, history) where
        history is a list of (iteration, best_score_so_far) tuples.
    """
    rng = np.random.default_rng(seed)
    positions = random_population(pop_size, rng)
    scores = np.full(pop_size, -np.inf)
    hp_cache = [None] * pop_size

    history = []
    best_score = -np.inf
    best_position = positions[0].copy()
    best_hp = None

    for it in range(max_iter):
        # Evaluate current pack.
        for i in range(pop_size):
            score, hp = fitness_fn(positions[i])
            scores[i] = score
            hp_cache[i] = hp
            print(f"[GWO] iter {it + 1}/{max_iter} wolf {i + 1}/{pop_size} "
                  f"macro_f1={score:.4f} hp={hp}")

        # Rank wolves: alpha = best, beta = 2nd best, delta = 3rd best.
        order = np.argsort(-scores)  # descending, since we maximize
        alpha_pos = positions[order[0]]
        beta_pos = positions[order[1 % pop_size]]
        delta_pos = positions[order[2 % pop_size]]

        if scores[order[0]] > best_score:
            best_score = scores[order[0]]
            best_position = positions[order[0]].copy()
            best_hp = hp_cache[order[0]]
        history.append((it, best_score))

        a = 2.0 - it * (2.0 / max_iter)  # linearly decreases 2 -> 0

        new_positions = np.empty_like(positions)
        for i in range(pop_size):
            leader_positions = (alpha_pos, beta_pos, delta_pos)
            x_estimates = []
            for leader_pos in leader_positions:
                r1, r2 = rng.random(DIM), rng.random(DIM)
                A = 2 * a * r1 - a          # exploration/exploitation coefficient
                C = 2 * r2                  # random weight on the leader's position
                D = np.abs(C * leader_pos - positions[i])
                x_estimates.append(leader_pos - A * D)
            new_positions[i] = np.mean(x_estimates, axis=0)

        positions = np.clip(new_positions, LOWER, UPPER)

    return best_position, best_score, best_hp, history


# ---------------------------------------------------------------------------
# Whale Optimization Algorithm (WOA) -- Mirjalili & Lewis, 2016
# ---------------------------------------------------------------------------

def whale_optimization_algorithm(fitness_fn, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=RANDOM_STATE):
    """Minimal Whale Optimization Algorithm over a continuous, bounded space.

    Standard WOA: each whale updates its position relative to the current
    best solution ("bubble-net feeding"), choosing between three moves each
    iteration:
      - shrinking encirclement of the best whale (|A| < 1, p < 0.5),
      - a random search using a randomly chosen whale instead of the best
        one (|A| >= 1, p < 0.5), which drives exploration, or
      - a logarithmic spiral update around the best whale (p >= 0.5), which
        mimics the bubble-net attack.
    `a` decays linearly from 2 to 0 over the iterations, shrinking the
    encirclement radius as the search progresses.

    Args:
        fitness_fn: callable position -> (score, hyperparams_dict), higher
            score is better (macro-F1, maximized).
        pop_size: number of whales.
        max_iter: number of iterations.
        seed: RNG seed for reproducibility.

    Returns:
        (best_position, best_score, best_hyperparams, history) where
        history is a list of (iteration, best_score_so_far) tuples.
    """
    rng = np.random.default_rng(seed)
    positions = random_population(pop_size, rng)

    history = []
    best_score = -np.inf
    best_position = positions[0].copy()
    best_hp = None

    b = 1.0  # spiral shape constant

    for it in range(max_iter):
        scores = np.full(pop_size, -np.inf)
        for i in range(pop_size):
            score, hp = fitness_fn(positions[i])
            scores[i] = score
            print(f"[WOA] iter {it + 1}/{max_iter} whale {i + 1}/{pop_size} "
                  f"macro_f1={score:.4f} hp={hp}")
            if score > best_score:
                best_score = score
                best_position = positions[i].copy()
                best_hp = hp
        history.append((it, best_score))

        a = 2.0 - it * (2.0 / max_iter)   # linearly decreases 2 -> 0
        a2 = -1.0 + it * (-1.0 / max_iter)  # linearly decreases -1 -> -2, controls spiral l

        new_positions = np.empty_like(positions)
        for i in range(pop_size):
            r1, r2 = rng.random(DIM), rng.random(DIM)
            A = 2 * a * r1 - a
            C = 2 * r2
            p = rng.random()

            if p < 0.5:
                if np.abs(A).mean() < 1.0:
                    # Encircle the current best whale.
                    D = np.abs(C * best_position - positions[i])
                    new_positions[i] = best_position - A * D
                else:
                    # Explore: move relative to a random whale in the pack.
                    rand_whale = positions[rng.integers(pop_size)]
                    D = np.abs(C * rand_whale - positions[i])
                    new_positions[i] = rand_whale - A * D
            else:
                # Spiral update around the best whale (bubble-net attack).
                l = rng.uniform(-1.0, 1.0)
                D_best = np.abs(best_position - positions[i])
                new_positions[i] = (
                    D_best * np.exp(b * l) * np.cos(2 * np.pi * l) + best_position
                )

        positions = np.clip(new_positions, LOWER, UPPER)

    return best_position, best_score, best_hp, history


# ---------------------------------------------------------------------------
# Final retrain + evaluate on the real test set, for a tuned hyperparameter set
# ---------------------------------------------------------------------------

def retrain_and_evaluate_best(name, hyperparams, X_train, y_train, X_test, y_test):
    """Retrain a model with `hyperparams` on the FULL training set for
    FINAL_EPOCHS, then evaluate it on the held-out test set via
    cnn_model.evaluate_model() (same metric/report/confusion-matrix format
    as every other model in this project).

    This is the only place a GWO/WOA-tuned model ever touches the test set,
    and it only does so once its hyperparameters are already fixed.
    """
    print(f"\n=== Retraining final '{name}' model with tuned hyperparams: {hyperparams} ===")
    model, history = train_and_evaluate(
        X_train, y_train, X_test, y_test,
        hyperparams=hyperparams,
        epochs=FINAL_EPOCHS,
        batch_size=BATCH_SIZE,
        model_name=name,
    )
    return model, history


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    keras.utils.set_random_seed(RANDOM_SEED)

    print(f"Loading processed data from {PROCESSED_DATA_DIR} ...")
    X_train_raw, y_train_raw, X_test_raw, y_test_raw = load_data()
    print(f"Train: {X_train_raw.shape}, Test: {X_test_raw.shape}")

    # Carve a validation split out of the TRAINING data only, for use during
    # the GWO/WOA search. The real test set (X_test_raw/y_test_raw) is never
    # used until retrain_and_evaluate_best(), after hyperparameters are
    # already fixed -- this avoids tuning against the test set.
    #
    # NOTE / caveat: cnn_model.load_data() returns pre-split, pre-processed
    # arrays without patient/subject IDs attached, so this split is a
    # stratified-by-label random split of the training beats rather than a
    # further patient-independent split. That matches what's available from
    # the existing preprocessing.py output; if patient IDs are exposed
    # alongside X_train/y_train in the future, prefer a patient-independent
    # split here too, for consistency with the project's evaluation protocol.
    X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
        X_train_raw, y_train_raw,
        test_size=VAL_FRACTION,
        random_state=RANDOM_STATE,
        stratify=y_train_raw,
    )

    X_tr = prepare_inputs(X_tr_raw)
    X_val = prepare_inputs(X_val_raw)
    X_test = prepare_inputs(X_test_raw)
    input_shape = X_tr.shape[1:]
    print(f"Search train: {X_tr.shape}, search val: {X_val.shape}, "
          f"held-out test (untouched during search): {X_test.shape}")

    fitness_fn = make_fitness_fn(X_tr, y_tr, X_val, y_val, input_shape)

    # --- Grey Wolf Optimizer ------------------------------------------------
    print("\n" + "=" * 70)
    print("Running Grey Wolf Optimizer (GWO) hyperparameter search")
    print("=" * 70)
    t0 = time.time()
    gwo_best_pos, gwo_best_val_f1, gwo_best_hp, gwo_history = grey_wolf_optimizer(
        fitness_fn, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=RANDOM_STATE,
    )
    gwo_search_time = time.time() - t0
    print(f"GWO best validation macro-F1: {gwo_best_val_f1:.4f}, "
          f"hyperparams: {gwo_best_hp} (search took {gwo_search_time:.1f}s)")

    # --- Whale Optimization Algorithm --------------------------------------
    print("\n" + "=" * 70)
    print("Running Whale Optimization Algorithm (WOA) hyperparameter search")
    print("=" * 70)
    t0 = time.time()
    woa_best_pos, woa_best_val_f1, woa_best_hp, woa_history = whale_optimization_algorithm(
        fitness_fn, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=RANDOM_STATE + 1,
    )
    woa_search_time = time.time() - t0
    print(f"WOA best validation macro-F1: {woa_best_val_f1:.4f}, "
          f"hyperparams: {woa_best_hp} (search took {woa_search_time:.1f}s)")

    # --- Retrain best configs on the full training set, evaluate on test --
    retrain_and_evaluate_best(
        "gwo_tuned_cnn", gwo_best_hp, X_train_raw, y_train_raw, X_test_raw, y_test_raw,
    )
    retrain_and_evaluate_best(
        "woa_tuned_cnn", woa_best_hp, X_train_raw, y_train_raw, X_test_raw, y_test_raw,
    )

    # --- Final comparison summary ------------------------------------------
    # The plain-Adam baseline is NOT re-run here -- src/cnn_model.py already
    # produces it (results/cnn_baseline_results.txt / results/cnn_results.txt
    # depending on the model_name used when it was run). We just try to read
    # its accuracy back out of that file so the summary is self-contained;
    # if it isn't found, we say so explicitly rather than guessing a number.
    baseline_acc = _read_baseline_accuracy(RESULTS_DIR)

    summary_lines = []
    summary_lines.append("Metaheuristic (GWO/WOA) vs standard Adam CNN tuning - comparison")
    summary_lines.append("=" * 70)
    summary_lines.append("")
    summary_lines.append(
        "Baseline (plain Adam, default hyperparams, from src/cnn_model.py):"
    )
    if baseline_acc is not None:
        summary_lines.append(f"  Test accuracy: {baseline_acc:.4f}")
    else:
        summary_lines.append(
            "  Not found in results/ -- run src/cnn_model.py first and re-run "
            "this script's summary step, or compare manually against its "
            "results/cnn_baseline_results.txt."
        )
    summary_lines.append("")
    summary_lines.append("GWO-tuned CNN:")
    summary_lines.append(f"  Search validation macro-F1: {gwo_best_val_f1:.4f}")
    summary_lines.append(f"  Tuned hyperparameters: {gwo_best_hp}")
    summary_lines.append("  Final test-set metrics: see results/gwo_tuned_cnn_results.txt")
    summary_lines.append("")
    summary_lines.append("WOA-tuned CNN:")
    summary_lines.append(f"  Search validation macro-F1: {woa_best_val_f1:.4f}")
    summary_lines.append(f"  Tuned hyperparameters: {woa_best_hp}")
    summary_lines.append("  Final test-set metrics: see results/woa_tuned_cnn_results.txt")
    summary_lines.append("")
    summary_lines.append(
        "Research question: is metaheuristic tuning worth it here?\n"
        "Compare the test accuracy / macro-F1 in results/cnn_baseline_results.txt\n"
        "against results/gwo_tuned_cnn_results.txt and results/woa_tuned_cnn_results.txt.\n"
        "If GWO/WOA test performance is not meaningfully better than the plain-Adam\n"
        "baseline given the extra compute (36 short trainings each on top of the\n"
        "final retrain), the practical answer for this dataset/model size is likely\n"
        "'no, not worth it' -- but let the actual numbers in those three files decide."
    )

    summary_text = "\n".join(summary_lines)
    print("\n" + summary_text)

    os.makedirs(RESULTS_DIR, exist_ok=True)
    summary_path = os.path.join(RESULTS_DIR, "metaheuristic_comparison.txt")
    with open(summary_path, "w") as f:
        f.write(summary_text + "\n")
    print(f"\nSaved comparison summary to {summary_path}")


def _read_baseline_accuracy(results_dir):
    """Best-effort scrape of the plain-Adam baseline's test accuracy out of
    the results file cnn_model.py's evaluate_model()/train_and_evaluate()
    already wrote (results/cnn_results.txt, since evaluate_model() is called
    there with name="CNN"; older runs may instead have written
    results/cnn_baseline_results.txt if model_name was used as the report
    name). Returns a float, or None if neither file is found/parseable.
    """
    for fname in ("cnn_results.txt", "cnn_baseline_results.txt"):
        path = Path(results_dir) / fname
        if not path.exists():
            continue
        try:
            for line in path.read_text().splitlines():
                if line.strip().lower().startswith("accuracy:"):
                    return float(line.split(":", 1)[1].strip())
        except (OSError, ValueError):
            continue
    return None


if __name__ == "__main__":
    main()


## 4. Download MIT-BIH and build the preprocessed dataset

Downloads the ~100MB MIT-BIH Arrhythmia Database into `data/raw/`, then filters/segments/labels every beat and saves `data/processed/*.npy` (including the RR-interval features used by the SVM/RF baseline).

In [ ]:
!python src/download_data.py


In [ ]:
!python src/preprocessing.py


## 5. Model 1 — SVM / Random Forest baseline (optional here)

Pure scikit-learn — this also runs fine on a laptop CPU, so skip this cell if you already ran it locally. Included so this notebook can be a complete, standalone run of the whole project.

In [ ]:
!python src/svm_baseline.py


## 6. Model 2 — 1D-CNN baseline

Trains the plain Adam-optimized CNN. Writes `results/cnn_baseline_results.txt`, `models/cnn_baseline.keras`, and per-epoch metrics to `logs/cnn_baseline_training_log.csv`.

In [ ]:
!python src/cnn_model.py


## 7. GWO / WOA metaheuristic hyperparameter search

Tunes the CNN's hyperparameters with Grey Wolf Optimizer and Whale Optimization Algorithm, retrains the best config from each on the full training set, and evaluates on the real test set. **This is the slowest cell** — see `metaheuristic_tuning.py`'s docstring for the exact compute-cost math (roughly 30-60 minutes on a T4). Reduce `POP_SIZE`/`MAX_ITER` in that file first if you're close to a session time limit.

In [ ]:
!python src/metaheuristic_tuning.py


## 7b. Grid-search control for the GWO/WOA experiment

Same CNN, same train/validation split, same macro-F1 fitness and same per-candidate budget (`FITNESS_EPOCHS`, patience) as the GWO/WOA search, but over a fixed 36-point grid (learning rate x base filters x kernel size). Retrains the best config for 50 epochs and writes `results/grid_tuned_cnn_results.txt`. Roughly 15-30 minutes on a T4.

In [ ]:
%%writefile src/grid_search_tuning.py
"""
grid_search_tuning.py

Control experiment for the GWO/WOA side experiment in
src/metaheuristic_tuning.py: does a plain grid search, given the SAME
compute budget, find 1D-CNN hyperparameters as good as the metaheuristics?
If it does, the extra machinery of GWO/WOA isn't buying anything on this
dataset/model size.

This script does NOT redefine the CNN, the data pipeline, or the fitness
function. It reuses, unchanged:
    from cnn_model.py:
        build_cnn_model(), load_data(), prepare_inputs(),
        compute_class_weights(), train_and_evaluate()
    from metaheuristic_tuning.py:
        make_fitness_fn() -- the exact macro-F1-on-validation fitness used
            to score every GWO/WOA candidate (internally calls
            build_cnn_model() and compute_class_weights()),
        decode_position() -- the position-vector -> hyperparams mapping,
        retrain_and_evaluate_best() -- the final full-training-set retrain
            via train_and_evaluate(),
        FITNESS_EPOCHS, FITNESS_PATIENCE (baked into make_fitness_fn()),
        BATCH_SIZE, FINAL_EPOCHS, VAL_FRACTION, RANDOM_STATE.
The train/validation split is the same train_test_split() call, with the
same VAL_FRACTION / RANDOM_STATE / stratification, that
metaheuristic_tuning.main() makes, so every grid candidate is scored on
exactly the same beats as every GWO/WOA candidate.

--------------------------------------------------------------------------
RUN THIS ON KAGGLE (or any machine with a real TensorFlow+GPU install) -
NOT in this local dev environment.
--------------------------------------------------------------------------
Same constraint as cnn_model.py / metaheuristic_tuning.py.

Search space and compute cost:
    Grid over the three hyperparameters most likely to matter for this
    small CNN; everything else stays at cnn_model.py's baseline value
    (dropout_rate=0.3, dense_units=64, pool_size=2):
        learning_rate     in {3e-4, 1e-3, 3e-3}        (3 values)
        num_filters base  in {16, 32, 64}              (3 values; blocks
                                                        use base, 2x, 4x)
        kernel_size       in {3, 5, 7, 9}              (4 values)
    3 * 3 * 4 = 36 candidates, each trained for up to FITNESS_EPOCHS = 8
    epochs with EarlyStopping(patience=FITNESS_PATIENCE = 3) -- exactly the
    same per-candidate budget, and the same total number of short
    trainings (POP_SIZE * MAX_ITER = 36), as ONE of the GWO/WOA runs.
    Plus 1 final retrain of the best config for FINAL_EPOCHS = 50 epochs.
    The grid contains cnn_model.py's baseline config (lr=1e-3, base 32,
    kernel 7) as one of its points.
    Budget roughly 15-30 minutes on a Kaggle T4/P100.

Outputs:
    results/grid_tuned_cnn_results.txt   (same format as every other model,
                                          written by cnn_model.evaluate_model)
    results/grid_search_scores.txt       (validation macro-F1 of every grid
                                          point, sorted best-first)
    models/grid_tuned_cnn.keras, logs/grid_tuned_cnn_training_log.csv

Usage (on Kaggle or a local machine with TensorFlow installed):
    python src/grid_search_tuning.py
"""

import os
import sys
import time
import itertools
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split

from tensorflow import keras

# cnn_model.py and metaheuristic_tuning.py live next to this file in src/.
sys.path.insert(0, str(Path(__file__).resolve().parent))

from cnn_model import (  # noqa: E402
    PROCESSED_DATA_DIR,
    RESULTS_DIR,
    RANDOM_SEED,
    load_data,
    prepare_inputs,
)
from metaheuristic_tuning import (  # noqa: E402
    FITNESS_EPOCHS,
    FITNESS_PATIENCE,
    VAL_FRACTION,
    RANDOM_STATE,
    decode_position,
    make_fitness_fn,
    retrain_and_evaluate_best,
)

# ---------------------------------------------------------------------------
# Grid
# ---------------------------------------------------------------------------

LEARNING_RATES = (3e-4, 1e-3, 3e-3)
FILTER_BASES = (16, 32, 64)
KERNEL_SIZES = (3, 5, 7, 9)

# Hyperparameters not searched here are pinned to cnn_model.py's baseline.
FIXED_DROPOUT_RATE = 0.3
FIXED_DENSE_UNITS = 64


def grid_positions():
    """Yield every grid point as a position vector in metaheuristic_tuning's
    encoding ([log10_lr, dropout_rate, num_filters_base, dense_units,
    kernel_size]), so it can be scored by the unchanged make_fitness_fn().
    """
    for lr, base, kernel in itertools.product(LEARNING_RATES, FILTER_BASES, KERNEL_SIZES):
        yield np.array([
            np.log10(lr), FIXED_DROPOUT_RATE, base, FIXED_DENSE_UNITS, kernel,
        ], dtype=float)


def _check_decodes_exactly(position):
    """Guard against decode_position() silently clipping/rounding a grid
    point to something other than what the grid says it is (e.g. if
    metaheuristic_tuning.BOUNDS were narrowed later).
    """
    hp = decode_position(position)
    expected_lr = 10.0 ** position[0]
    assert np.isclose(hp["learning_rate"], expected_lr), (hp, position)
    assert hp["num_filters"][0] == int(position[2]), (hp, position)
    assert hp["kernel_size"] == int(position[4]), (hp, position)
    assert np.isclose(hp["dropout_rate"], position[1]), (hp, position)
    assert hp["dense_units"] == int(position[3]), (hp, position)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    keras.utils.set_random_seed(RANDOM_SEED)

    print(f"Loading processed data from {PROCESSED_DATA_DIR} ...")
    X_train_raw, y_train_raw, X_test_raw, y_test_raw = load_data()
    print(f"Train: {X_train_raw.shape}, Test: {X_test_raw.shape}")

    # Identical split to metaheuristic_tuning.main(): validation carved out of
    # the TRAINING data only; the test set is untouched until the final retrain.
    X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
        X_train_raw, y_train_raw,
        test_size=VAL_FRACTION,
        random_state=RANDOM_STATE,
        stratify=y_train_raw,
    )

    X_tr = prepare_inputs(X_tr_raw)
    X_val = prepare_inputs(X_val_raw)
    input_shape = X_tr.shape[1:]
    print(f"Search train: {X_tr.shape}, search val: {X_val.shape}")

    fitness_fn = make_fitness_fn(X_tr, y_tr, X_val, y_val, input_shape)

    positions = list(grid_positions())
    for pos in positions:
        _check_decodes_exactly(pos)
    n = len(positions)

    print("\n" + "=" * 70)
    print(f"Running grid search over {n} candidates "
          f"(up to {FITNESS_EPOCHS} epochs each, patience {FITNESS_PATIENCE})")
    print("=" * 70)

    t0 = time.time()
    scored = []
    for i, pos in enumerate(positions):
        score, hp = fitness_fn(pos)
        scored.append((score, hp))
        print(f"[GRID] {i + 1}/{n} macro_f1={score:.4f} hp={hp}")
    search_time = time.time() - t0

    # Stable sort keeps the earlier grid point on ties.
    scored.sort(key=lambda s: -s[0])
    best_val_f1, best_hp = scored[0]
    print(f"Grid best validation macro-F1: {best_val_f1:.4f}, "
          f"hyperparams: {best_hp} (search took {search_time:.1f}s)")

    os.makedirs(RESULTS_DIR, exist_ok=True)
    scores_path = os.path.join(RESULTS_DIR, "grid_search_scores.txt")
    with open(scores_path, "w") as f:
        f.write(f"Grid search validation macro-F1 ({n} candidates, "
                f"FITNESS_EPOCHS={FITNESS_EPOCHS}, patience={FITNESS_PATIENCE}, "
                f"search took {search_time:.1f}s)\n\n")
        for score, hp in scored:
            f.write(f"{score:.4f}  {hp}\n")
    print(f"Saved per-candidate scores to {scores_path}")

    # Retrains for FINAL_EPOCHS (50) on the full training set and writes
    # results/grid_tuned_cnn_results.txt via cnn_model.evaluate_model().
    retrain_and_evaluate_best(
        "grid_tuned_cnn", best_hp, X_train_raw, y_train_raw, X_test_raw, y_test_raw,
    )

    print("\nDone. Compare results/grid_tuned_cnn_results.txt against "
          "results/gwo_tuned_cnn_results.txt and results/woa_tuned_cnn_results.txt.")


if __name__ == "__main__":
    main()


In [ ]:
!python src/grid_search_tuning.py


## 8. Model 3 — CNN-LSTM hybrid

Builds 5-beat sequences within each patient record (never across records), trains the CNN-LSTM, and evaluates on the test set.

In [ ]:
!python src/cnn_lstm_model.py


## 9. Package everything for download

Zips `results/`, `models/`, and `logs/` so you can pull the trained models and every results file back to your machine from the notebook's Output panel.

In [ ]:
!zip -r outputs.zip results models logs
print("Done. Download outputs.zip from the notebook's file browser (right panel).")


## 10. Print a quick side-by-side accuracy summary

Best-effort scrape of every `results/*_results.txt` file's headline accuracy, for a quick look before you dig into the full classification reports.

In [ ]:
import glob

for path in sorted(glob.glob("results/*_results.txt")):
    acc_line = None
    with open(path) as f:
        for line in f:
            if line.strip().lower().startswith("accuracy:"):
                acc_line = line.strip()
                break
    print(f"{path:45s} {acc_line or '(no accuracy line found)'}")
